# CocoStudio Cloud GPU Character Factory
Use a GPU runtime. This notebook processes the next job and pushes its result back to GitHub.

In [ ]:
from getpass import getpass
from google.colab import userdata

try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = getpass('Paste your NEW GitHub token (input is hidden): ')

if not GITHUB_TOKEN:
    raise RuntimeError('A GitHub token is required to read and write cloud jobs.')

REPOSITORY_URL = 'https://github.com/SK1111111111/CocoStudio-AI-Jobs.git'
BRANCH = 'main'
AUTH_REPOSITORY_URL = REPOSITORY_URL.replace('https://', f'https://x-access-token:{GITHUB_TOKEN}@')
print('GitHub authentication configured securely.')


In [ ]:
import shutil
import subprocess
import time
import torch

started_at = time.time()
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not active. Select Runtime > Change runtime type > T4 GPU, then Run all again.')
print('Stage 1/4: GPU ready:', torch.cuda.get_device_name(0), flush=True)

print('Stage 2/4: Downloading CocoStudio cloud job...', flush=True)
shutil.rmtree('/content/CocoStudio-AI-Jobs', ignore_errors=True)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, AUTH_REPOSITORY_URL, '/content/CocoStudio-AI-Jobs'], check=True)

print('Stage 3/4: Downloading Hunyuan3D source...', flush=True)
shutil.rmtree('/content/Hunyuan3D-2', ignore_errors=True)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git', '/content/Hunyuan3D-2'], check=True)
%cd /content/Hunyuan3D-2

print('Stage 4/4: Installing Hunyuan3D dependencies. The first run can take 10-25 minutes...', flush=True)
subprocess.run(['pip', 'install', '-r', 'requirements.txt'], check=True)
subprocess.run(['pip', 'install', '-e', '.'], check=True)
print(f'Installation complete in {(time.time() - started_at) / 60:.1f} minutes.', flush=True)


In [ ]:
%cd /content/CocoStudio-AI-Jobs
!git config user.name 'CocoStudio Cloud GPU'
!git config user.email 'cocostudio-cloud@local'
!python run_cloud_factory.py
